# 05 — Task 2.5: SCD Type 2 for Customer Risk Profile

**Target Table:** `bfsi.silver_layer.dim_customer_risk`

**Objective:** Implement Slowly Changing Dimension Type 2 to track historical changes  
to customer risk attributes for Basel III regulatory compliance.

**Tracked Columns:**
- `risk_rating` — LOW / MEDIUM / HIGH
- `kyc_status` — VERIFIED / PENDING
- `account_type` — SAVINGS / CURRENT / NRE / NRO

**SCD2 Pattern:**
- When a tracked column changes → close the old record (`is_current = FALSE`, set `effective_end_date`)
- Insert a new record (`is_current = TRUE`, `effective_start_date = now`, `effective_end_date = 9999-12-31`)
- Unchanged customers → no action

> **Regulatory:** Risk profile history is required for Basel III capital calculation accuracy (7-year retention).

---
## Cell 1 — Configuration & Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, BooleanType, TimestampType
)
from datetime import datetime, date

# ── Catalog / Schema ───────────────────────────────────────────────────
CATALOG       = "bfsi"
SILVER_SCHEMA = "silver_layer"
BRONZE_SCHEMA = "bronze_layer"

# ── Target SCD2 table ──────────────────────────────────────────────────
SCD2_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_risk"

# ── SCD2 tracked columns ───────────────────────────────────────────────
TRACKED_COLS = ["risk_rating", "kyc_status", "account_type"]

# ── End-of-time date for active records ────────────────────────────────
EOT_DATE = "9999-12-31"

print(f"Task 2.5: SCD Type 2 — Customer Risk Profile")
print(f"Tracked cols : {TRACKED_COLS}")
print(f"Target table : {SCD2_TABLE}")
print(f"Timestamp    : {datetime.now().isoformat()}")

---
## Cell 2 — Create Test CSV Snapshots

Two CSV snapshots simulating customer data at different points in time:  
- **Snapshot 1** (2024-01-15): Initial customer data  
- **Snapshot 2** (2024-08-20): Some customers have changed `risk_rating`, `kyc_status`, or `account_type`

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  TEST DATA: Two CSV snapshots with different risk_ratings
#  These simulate the bronze.customers table at two points in time
# ══════════════════════════════════════════════════════════════════════════

# ── Snapshot 1: 2024-01-15 (initial load) ──────────────────────────────
snapshot_1_data = [
    ("CUST001", "LOW",    "VERIFIED", "SAVINGS"),
    ("CUST002", "MEDIUM", "VERIFIED", "CURRENT"),
    ("CUST003", "HIGH",   "PENDING",  "SAVINGS"),
    ("CUST004", "LOW",    "VERIFIED", "NRE"),
    ("CUST005", "MEDIUM", "VERIFIED", "SAVINGS"),
    ("CUST006", "LOW",    "PENDING",  "CURRENT"),
    ("CUST007", "HIGH",   "VERIFIED", "NRO"),
    ("CUST008", "LOW",    "VERIFIED", "SAVINGS"),
]

# ── Snapshot 2: 2024-08-20 (changes applied) ──────────────────────────
# Changes:
#   CUST001: risk_rating LOW → HIGH       (suspected fraud activity)
#   CUST003: kyc_status  PENDING → VERIFIED (KYC completed)
#   CUST005: risk_rating MEDIUM → LOW     (cleared after investigation)
#   CUST006: account_type CURRENT → SAVINGS + kyc_status PENDING → VERIFIED
#   CUST002, CUST004, CUST007, CUST008: NO CHANGE
snapshot_2_data = [
    ("CUST001", "HIGH",   "VERIFIED", "SAVINGS"),   # risk_rating changed
    ("CUST002", "MEDIUM", "VERIFIED", "CURRENT"),   # no change
    ("CUST003", "HIGH",   "VERIFIED", "SAVINGS"),   # kyc_status changed
    ("CUST004", "LOW",    "VERIFIED", "NRE"),       # no change
    ("CUST005", "LOW",    "VERIFIED", "SAVINGS"),   # risk_rating changed
    ("CUST006", "LOW",    "VERIFIED", "SAVINGS"),   # account_type + kyc changed
    ("CUST007", "HIGH",   "VERIFIED", "NRO"),       # no change
    ("CUST008", "LOW",    "VERIFIED", "SAVINGS"),   # no change
]

snapshot_schema = StructType([
    StructField("customer_id",  StringType(), False),
    StructField("risk_rating",  StringType(), True),
    StructField("kyc_status",   StringType(), True),
    StructField("account_type", StringType(), True),
])

df_snapshot_1 = spark.createDataFrame(snapshot_1_data, schema=snapshot_schema)
df_snapshot_2 = spark.createDataFrame(snapshot_2_data, schema=snapshot_schema)

print("Snapshot 1 (2024-01-15) — initial load:")
display(df_snapshot_1)
print("\nSnapshot 2 (2024-08-20) — with changes:")
display(df_snapshot_2)

---
## Cell 3 — Create SCD2 Target Table

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  CREATE dim_customer_risk SCD2 TABLE
#  Schema includes surrogate key, tracked columns, and SCD2 metadata
# ══════════════════════════════════════════════════════════════════════════

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SCD2_TABLE} (
        -- Surrogate key (unique per version of a customer record)
        sk_customer_risk    BIGINT GENERATED ALWAYS AS IDENTITY,
        -- Business key
        customer_id         STRING      NOT NULL,
        -- Tracked SCD2 columns
        risk_rating         STRING,
        kyc_status          STRING,
        account_type        STRING,
        -- SCD2 metadata
        effective_start_date DATE       NOT NULL,
        effective_end_date   DATE       NOT NULL,
        is_current           BOOLEAN    NOT NULL,
        -- Audit columns
        _load_ts             TIMESTAMP,
        _change_reason       STRING
    )
    USING DELTA
    COMMENT 'SCD2 dim — customer risk profile history (Basel III regulatory retention)'
    TBLPROPERTIES (
        'delta.deletedFileRetentionDuration' = 'interval 2555 days',
        'delta.logRetentionDuration'         = 'interval 2555 days',
        'delta.enableChangeDataFeed'         = 'true'
    )
""")

print(f"SCD2 table created: {SCD2_TABLE}")
print(f"Schema:")
spark.read.table(SCD2_TABLE).printSchema()

---
## Cell 4 — Initial Load (Snapshot 1)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  INITIAL LOAD — Snapshot 1 (2024-01-15)
#  All records are inserted as current (is_current = TRUE)
#  effective_start_date = 2024-01-15, effective_end_date = 9999-12-31
# ══════════════════════════════════════════════════════════════════════════

SNAPSHOT_1_DATE = "2024-01-15"

df_initial = (
    df_snapshot_1
    .withColumn("effective_start_date", F.to_date(F.lit(SNAPSHOT_1_DATE)))
    .withColumn("effective_end_date",   F.to_date(F.lit(EOT_DATE)))
    .withColumn("is_current",           F.lit(True))
    .withColumn("_load_ts",             F.current_timestamp())
    .withColumn("_change_reason",       F.lit("INITIAL_LOAD"))
)

# Write initial load (append since table may already have data from prior runs)
# For clean test: overwrite
df_initial.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SCD2_TABLE)

print(f"Initial load complete: {spark.read.table(SCD2_TABLE).count()} records")
print(f"All records current as of {SNAPSHOT_1_DATE}")
display(spark.read.table(SCD2_TABLE).orderBy("customer_id"))

---
## Cell 5 — SCD2 MERGE Logic (Reusable Function)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  SCD TYPE 2 MERGE FUNCTION
#  Reusable function that applies SCD2 logic using Delta MERGE INTO
#
#  Logic:
#    1. MATCH on customer_id + is_current = TRUE
#    2. If any tracked column changed → close old record (set end date,
#       is_current = FALSE)
#    3. Insert new record with updated values + is_current = TRUE
#    4. If no tracked column changed → no action (skip)
# ══════════════════════════════════════════════════════════════════════════

def apply_scd2_merge(df_incoming, snapshot_date, tracked_cols, target_table):
    """
    Apply SCD Type 2 merge on dim_customer_risk.
    
    Parameters
    ----------
    df_incoming : DataFrame
        New snapshot with columns: customer_id + tracked_cols
    snapshot_date : str
        Date of the incoming snapshot (YYYY-MM-DD)
    tracked_cols : list[str]
        Columns to track for changes
    target_table : str
        Fully qualified Delta table name
    """
    from delta.tables import DeltaTable
    
    dt_target = DeltaTable.forName(spark, target_table)

    # ── Step 1: Identify changed records ──────────────────────────────
    # Join incoming with current records to detect changes
    df_current = spark.read.table(target_table).filter(F.col("is_current") == True)
    
    # Build change detection condition
    # A record has changed if ANY tracked column differs
    change_condition = F.lit(False)
    for col in tracked_cols:
        change_condition = change_condition | (
            F.coalesce(F.col(f"incoming.{col}"), F.lit("__NULL__")) !=
            F.coalesce(F.col(f"current.{col}"), F.lit("__NULL__"))
        )
    
    df_changes = (
        df_incoming.alias("incoming")
        .join(
            df_current.alias("current"),
            on="customer_id",
            how="inner"
        )
        .filter(change_condition)
        .select("incoming.*")
    )
    
    changed_count = df_changes.count()
    print(f"  Changed records detected: {changed_count}")
    
    if changed_count == 0:
        print("  No changes — skipping merge.")
        return
    
    # ── Step 2: Build change reason ───────────────────────────────────
    # Create a description of what changed for each record
    change_reasons = []
    for col in tracked_cols:
        change_reasons.append(
            F.when(
                F.coalesce(F.col(f"incoming.{col}"), F.lit("__NULL__")) !=
                F.coalesce(F.col(f"current.{col}"), F.lit("__NULL__")),
                F.concat(
                    F.lit(f"{col}: "),
                    F.coalesce(F.col(f"current.{col}"), F.lit("NULL")),
                    F.lit(" → "),
                    F.coalesce(F.col(f"incoming.{col}"), F.lit("NULL"))
                )
            )
        )
    
    df_with_reason = (
        df_incoming.alias("incoming")
        .join(df_current.alias("current"), on="customer_id", how="inner")
        .filter(change_condition)
        .select(
            "incoming.*",
            F.concat_ws("; ", F.array_compact(F.array(*change_reasons))).alias("_change_detail")
        )
    )

    # ── Step 3: Prepare STAGED data for merge ─────────────────────────
    # Staged rows: the NEW version of changed records
    df_staged_inserts = (
        df_with_reason
        .select(
            F.col("customer_id"),
            *[F.col(c) for c in tracked_cols],
            F.to_date(F.lit(snapshot_date)).alias("effective_start_date"),
            F.to_date(F.lit(EOT_DATE)).alias("effective_end_date"),
            F.lit(True).alias("is_current"),
            F.current_timestamp().alias("_load_ts"),
            F.col("_change_detail").alias("_change_reason"),
        )
        # Mark these rows for insert (not update)
        .withColumn("_merge_key", F.lit(None).cast(StringType()))
    )
    
    # Also prepare the existing rows to be updated (close old record)
    df_staged_updates = (
        df_changes
        .select(F.col("customer_id").alias("_merge_key"))
    )
    
    # Union: updates (matched) + inserts (not matched)
    df_staged = (
        df_staged_inserts
        .unionByName(
            df_staged_updates
            .withColumn("customer_id", F.col("_merge_key"))
            .withColumn("risk_rating", F.lit(None))
            .withColumn("kyc_status", F.lit(None))
            .withColumn("account_type", F.lit(None))
            .withColumn("effective_start_date", F.lit(None).cast(DateType()))
            .withColumn("effective_end_date", F.lit(None).cast(DateType()))
            .withColumn("is_current", F.lit(None).cast(BooleanType()))
            .withColumn("_load_ts", F.lit(None).cast(TimestampType()))
            .withColumn("_change_reason", F.lit(None))
        )
    )

    # ── Step 4: MERGE INTO ────────────────────────────────────────────
    dt_target.alias("target").merge(
        df_staged.alias("source"),
        condition=(
            "target.customer_id = source._merge_key "
            "AND target.is_current = TRUE"
        )
    ).whenMatchedUpdate(
        # Close the old record
        set={
            "is_current": F.lit(False),
            "effective_end_date": F.date_sub(F.to_date(F.lit(snapshot_date)), 1),
        }
    ).whenNotMatchedInsert(
        # Insert new version
        condition="source._merge_key IS NULL",
        values={
            "customer_id":          F.col("source.customer_id"),
            "risk_rating":          F.col("source.risk_rating"),
            "kyc_status":           F.col("source.kyc_status"),
            "account_type":         F.col("source.account_type"),
            "effective_start_date": F.col("source.effective_start_date"),
            "effective_end_date":   F.col("source.effective_end_date"),
            "is_current":           F.col("source.is_current"),
            "_load_ts":             F.col("source._load_ts"),
            "_change_reason":       F.col("source._change_reason"),
        }
    ).execute()

    print(f"  MERGE complete. Closed {changed_count} old records, inserted {changed_count} new versions.")


print("SCD2 merge function defined: apply_scd2_merge()")

---
## Cell 6 — Apply SCD2 Merge (Snapshot 2)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  APPLY SCD2 MERGE — Snapshot 2 (2024-08-20)
#  Expected changes:
#    CUST001: risk_rating LOW → HIGH
#    CUST003: kyc_status PENDING → VERIFIED
#    CUST005: risk_rating MEDIUM → LOW
#    CUST006: account_type CURRENT → SAVINGS + kyc_status PENDING → VERIFIED
# ══════════════════════════════════════════════════════════════════════════

SNAPSHOT_2_DATE = "2024-08-20"

print(f"Applying SCD2 merge for snapshot: {SNAPSHOT_2_DATE}")
apply_scd2_merge(
    df_incoming=df_snapshot_2,
    snapshot_date=SNAPSHOT_2_DATE,
    tracked_cols=TRACKED_COLS,
    target_table=SCD2_TABLE,
)

print(f"\nFull SCD2 table after merge:")
display(
    spark.read.table(SCD2_TABLE)
    .orderBy("customer_id", "effective_start_date")
)

---
## Cell 7 — Verify SCD2 Results

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  VERIFICATION: Check SCD2 correctness
# ══════════════════════════════════════════════════════════════════════════

df_scd2 = spark.read.table(SCD2_TABLE)

print("=" * 70)
print("  SCD2 VERIFICATION")
print("=" * 70)

# ── Total record counts ──
total_records = df_scd2.count()
current_records = df_scd2.filter(F.col("is_current") == True).count()
historical_records = df_scd2.filter(F.col("is_current") == False).count()

print(f"\n Total records   : {total_records}")
print(f" Current (active): {current_records}")
print(f" Historical      : {historical_records}")

# ── Verify expected: 8 customers, 4 changed = 8 + 4 = 12 total ──
assert total_records == 12, f"Expected 12 records, got {total_records}"
assert current_records == 8, f"Expected 8 current records, got {current_records}"
assert historical_records == 4, f"Expected 4 historical records, got {historical_records}"
print(" Record counts: PASS (8 current + 4 historical = 12 total)")

# ── Verify CUST001 has 2 records ──
cust001 = df_scd2.filter(F.col("customer_id") == "CUST001").orderBy("effective_start_date")
cust001_count = cust001.count()
assert cust001_count == 2, f"CUST001 should have 2 records, got {cust001_count}"
print(" CUST001 versions: PASS (2 records)")

print("\n CUST001 history:")
display(cust001)

# ── Verify unchanged customers have 1 record ──
for cid in ["CUST002", "CUST004", "CUST007", "CUST008"]:
    cnt = df_scd2.filter(F.col("customer_id") == cid).count()
    assert cnt == 1, f"{cid} should have 1 record (unchanged), got {cnt}"
print(" Unchanged customers: PASS (1 record each for CUST002, 004, 007, 008)")

print("\n ALL VERIFICATIONS PASSED")
print("=" * 70)

---
## Cell 8 — Historical Point-in-Time Query

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  HISTORICAL QUERY: What was CUST001's risk_rating on 2024-06-01?
#
#  Expected answer: "LOW"
#  Reasoning:
#    - CUST001 was initially LOW (effective 2024-01-15 to 2024-08-19)
#    - Changed to HIGH on 2024-08-20
#    - On 2024-06-01, the LOW record was still active
# ══════════════════════════════════════════════════════════════════════════

QUERY_DATE = "2024-06-01"
QUERY_CUSTOMER = "CUST001"

print(f"Query: What was {QUERY_CUSTOMER}'s risk_rating on {QUERY_DATE}?")
print("=" * 60)

# Point-in-time SCD2 query pattern:
#   effective_start_date <= query_date <= effective_end_date
df_pit = (
    spark.read.table(SCD2_TABLE)
    .filter(
        (F.col("customer_id") == QUERY_CUSTOMER) &
        (F.col("effective_start_date") <= QUERY_DATE) &
        (F.col("effective_end_date") >= QUERY_DATE)
    )
)

display(df_pit)

result = df_pit.collect()
if result:
    risk = result[0]["risk_rating"]
    print(f"\n Answer: {QUERY_CUSTOMER}'s risk_rating on {QUERY_DATE} was '{risk}'")
    assert risk == "LOW", f"Expected 'LOW', got '{risk}'"
    print(" Verification: PASS — correctly returned historical 'LOW' (not current 'HIGH')")
else:
    print(" ERROR: No record found for the query date")

print("\n" + "=" * 60)

# ── Also verify CURRENT risk_rating ──
print(f"\nCross-check: {QUERY_CUSTOMER}'s CURRENT risk_rating:")
df_current = (
    spark.read.table(SCD2_TABLE)
    .filter(
        (F.col("customer_id") == QUERY_CUSTOMER) &
        (F.col("is_current") == True)
    )
)
display(df_current)
current_risk = df_current.collect()[0]["risk_rating"]
print(f"Answer: Current risk_rating = '{current_risk}' (changed from LOW → HIGH on 2024-08-20)")

---
## Cell 9 — Production: Load from Bronze Customers

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  PRODUCTION MODE: Apply SCD2 from Bronze customers table
#  Uncomment and use this cell when running against real data
#  instead of the test snapshots above
# ══════════════════════════════════════════════════════════════════════════

# PRODUCTION_SNAPSHOT_DATE = datetime.now().strftime("%Y-%m-%d")

# # Read current customer data from Bronze
# df_bronze_customers = (
#     spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
#     .select("customer_id", "risk_rating", "kyc_status", "account_type")
# )
# 
# print(f"Production SCD2 merge — snapshot date: {PRODUCTION_SNAPSHOT_DATE}")
# print(f"Incoming records: {df_bronze_customers.count():,}")
# 
# apply_scd2_merge(
#     df_incoming=df_bronze_customers,
#     snapshot_date=PRODUCTION_SNAPSHOT_DATE,
#     tracked_cols=TRACKED_COLS,
#     target_table=SCD2_TABLE,
# )

print("Production SCD2 cell ready (uncomment to use with real Bronze data).")

---
## Cell 10 — Summary

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  TASK 2.5 COMPLETION SUMMARY
# ══════════════════════════════════════════════════════════════════════════

df_final = spark.read.table(SCD2_TABLE)

print("=" * 70)
print("  TASK 2.5: SCD TYPE 2 — CUSTOMER RISK PROFILE — COMPLETE")
print("=" * 70)

print(f"\n Table         : {SCD2_TABLE}")
print(f" Total records : {df_final.count()}")
print(f" Current       : {df_final.filter(F.col('is_current')==True).count()}")
print(f" Historical    : {df_final.filter(F.col('is_current')==False).count()}")

print(f"\n Tracked columns: {TRACKED_COLS}")
print(f" SCD2 metadata : effective_start_date, effective_end_date, is_current")
print(f" Change data feed: ENABLED")
print(f" Retention      : 7 years (2555 days) — Basel III compliant")

print("\n Test results:")
print("  ✓ Initial load: 8 customers loaded (Snapshot 1: 2024-01-15)")
print("  ✓ SCD2 merge : 4 changes detected and applied (Snapshot 2: 2024-08-20)")
print("  ✓ Historical query: CUST001 risk_rating on 2024-06-01 = 'LOW' (correct)")
print("  ✓ Current query  : CUST001 risk_rating now = 'HIGH' (correct)")
print("  ✓ Record counts  : 12 total (8 current + 4 historical)")

print(f"\n Timestamp: {datetime.now().isoformat()}")
print("=" * 70)

---

### SCD2 Query Patterns

```sql
-- Current risk profile
SELECT * FROM bfsi.silver_layer.dim_customer_risk
WHERE customer_id = 'CUST001' AND is_current = TRUE;

-- Point-in-time query (Basel III audit)
SELECT * FROM bfsi.silver_layer.dim_customer_risk
WHERE customer_id = 'CUST001'
  AND effective_start_date <= '2024-06-01'
  AND effective_end_date   >= '2024-06-01';

-- Full history for a customer
SELECT * FROM bfsi.silver_layer.dim_customer_risk
WHERE customer_id = 'CUST001'
ORDER BY effective_start_date;

-- All customers whose risk was upgraded
SELECT * FROM bfsi.silver_layer.dim_customer_risk
WHERE _change_reason LIKE '%risk_rating%'
  AND is_current = TRUE;
```

> **Regulatory note:** This SCD2 table satisfies Basel III requirements for historical risk  
> profile tracking. Change Data Feed is enabled for downstream CDC consumers.